# 03 — `@numfast/kernels`: a portable kernel library, not a compute core

This is the **WASM channel**, and it is a different artefact from the Python
package — not a wrapper around it. The same Rust kernels are compiled to
WebAssembly and published to npm.

What it is, stated before anything is run: **a kernel library.** There is **no
executor, no graph runtime, no IR dispatch and no buffer allocator inside the
`.wasm`**. Orchestration, buffer ownership and the type layer stay in the host.
Any statement below about scale is read from the shipped bytes or from the
package's own generated constants — not typed in.

Every cell drives Node through a subprocess, because the module is JavaScript
and a Python notebook cannot import it honestly.

In [1]:
import json
import subprocess
from pathlib import Path

ROOT = Path.cwd()
TS = ROOT / "numfast-native" / "ts"
DIST = TS / "dist"


def node_json(source):
    # Run a Node module snippet and return its JSON.
    #
    # Node may prepend warnings to stdout, so the payload is taken from the last
    # line that actually parses. A failure raises with stderr attached, rather
    # than leaving a half-built notebook whose outputs silently mean nothing.
    res = subprocess.run(["node", "--input-type=module", "-e", source],
                         cwd=TS, capture_output=True, text=True)
    for line in reversed([ln for ln in res.stdout.splitlines() if ln.strip()]):
        try:
            return json.loads(line)
        except json.JSONDecodeError:
            continue
    raise RuntimeError("node produced no JSON. rc=%s stdout=%s stderr=%s"
                       % (res.returncode, res.stdout[-500:], res.stderr[-500:]))


import numfast as nf
print("numfast", nf.__version__, "(the Python package; separate artefact)")
print("wasm package:", json.loads((TS / "package.json").read_text(encoding="utf-8"))["name"])

numfast 0.2.1 (the Python package; separate artefact)
wasm package: @numfast/kernels


## 1. The artefact, and whether the bytes are the ones the build recorded

`dist/BUILD.json` records the sha256, the byte count and the git commit the
`.wasm` was built from. `test/artifact.mjs` refuses to run a parity test against
a `.wasm` whose sha256 disagrees with it — "a test that cannot name the bytes it
validated is not a test, it is a print statement". That refusal is reproduced
here rather than assumed.

In [2]:
import hashlib

build = json.loads((DIST / "BUILD.json").read_text(encoding="utf-8"))
wasm = DIST / "numfast_native.wasm"
actual = hashlib.sha256(wasm.read_bytes()).hexdigest()

print("BUILD.json sha256:", build["sha256"])
print("on disk   sha256:", actual)
print("MATCH:", actual == build["sha256"])
print()
print(f"bytes         {build['bytes']:,}")
print(f"version       {build['version']}")
print(f"builtFrom     {build['builtFrom'][:7]}")

BUILD.json sha256: 16f50fe9b2ee9aee81eddcfb1c6ea2b9490a3e79955b5c7e0b0a3e7bcd4cc71b
on disk   sha256: 16f50fe9b2ee9aee81eddcfb1c6ea2b9490a3e79955b5c7e0b0a3e7bcd4cc71b
MATCH: True

bytes         196,719
version       0.2.1
builtFrom     32290bd


In [3]:
import subprocess

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=ROOT,
                      capture_output=True, text=True).stdout.strip()
print("HEAD           ", head[:40])
print("the .wasm was built from this exact commit:", head == build["builtFrom"])

HEAD            32290bd3fd51f140ef9a07591261424e04220307
the .wasm was built from this exact commit: True


## 2. Loading one module, once

`loadKernels()` reads the packaged `.wasm` and `BUILD.json` off disk. That is a
real Node dependency — it is the reason the bare entry is Node-only — and it is
why the browser-facing entry is a second module. **One module, loaded once**,
is the shape of the whole API: there is nothing to rebuild per call.

In [4]:
HELPER = r'''
import { loadKernels, WRAPPED, TOTAL_EXPORTS, SCOPE, exportsOf } from "./dist/index.js";
const k = await loadKernels();
const ex = exportsOf(k);
const names = Object.keys(ex).sort();
const funcNames = names.filter((n) => n !== "memory");
const mem = ex.memory;
console.log(JSON.stringify({
  exports_total: names.length,
  function_exports: funcNames.length,
  memory_exports: names.filter((n) => n === "memory").length,
  import_count: WebAssembly.Module.imports(new WebAssembly.Module(
      (await import("node:fs")).readFileSync("./dist/numfast_native.wasm"))).length,
  initial_memory_pages: mem.buffer.byteLength / 65536,
  wrapped: WRAPPED.length,
  total_exports_const: TOTAL_EXPORTS,
  base: k.base,
  static_data_end: k.staticDataEnd,
  scope_statement: SCOPE.statement,
}));
'''

facts = node_json(HELPER)
for key in ("exports_total", "function_exports", "memory_exports",
            "import_count", "initial_memory_pages", "wrapped",
            "total_exports_const", "base", "static_data_end"):
    print(f"  {key:22s} {facts[key]}")

  exports_total          87
  function_exports       86
  memory_exports         1
  import_count           0
  initial_memory_pages   17
  wrapped                17
  total_exports_const    86
  base                   1114112
  static_data_end        1061411


**86 function exports, one memory, zero imports, 17 typed wrappers.** The
memory is the module's own, 17 pages × 64 KiB = 1,114,112 bytes, and it starts
with a stack pointer and an initialised data region — which is why the bridge
probes for `staticDataEnd` and allocates above it rather than assuming a
constant base.

In [5]:
print("86 functions :", facts["function_exports"] == 86)
print("one memory   :", facts["memory_exports"] == 1)
print("zero imports :", facts["import_count"] == 0)
print("17 wrapped   :", facts["wrapped"] == 17)

print()
print("SCOPE.statement, generated by the package from those two numbers:")
print()
for para in facts["scope_statement"].split(". "):
    print("  " + para.strip() + ".")

86 functions : True
one memory   : True
zero imports : True
17 wrapped   : True

SCOPE.statement, generated by the package from those two numbers:

  NumFast compiles its compute kernels to WebAssembly: an 86-function module with one memory and no imports, so the module itself instantiates in any host with a WebAssembly runtime.
  This package wraps 17 of those 86 kernels with typed wrappers and is a portable kernel library, not a compute core: orchestration, buffer management and the type layer stay in the host.
  Its own entry point is Node-only (it reads the packaged .wasm with node:fs); the browser-facing module is dist/bridge.js, which takes the bytes you give it.
  The remaining 69 kernels are reachable by name through callRaw and have no documented argument order..


That sentence is **generated** — `SCOPE.statement` is built in `index.ts` at
module load from `TOTAL_EXPORTS` and `WRAPPED.length`, so it cannot drift from
the two numbers it describes. The same two numbers are quoted in the package
README, in `docs/INSTALL.md` and in the root README, and
`tests/fast/test_doc_counters.py` checks all of those sites against the
generated values.

## 3. Repeated calls against the one loaded module

Every call below reuses the **same bridge and the same linear memory**. The
bridge keeps a bump cursor (`cur`) and hands out non-overlapping regions, so
repeated calls do not re-instantiate and do not allocate from the JS heap.

In [6]:
RUNS = r'''
import { loadKernels, mapI32, mapScalarI32, mapF64, adjacencyGather } from "./dist/index.js";
const k = await loadKernels();
const out = {};
const a = new Int32Array([1, 2, 3, 4]);
const b = new Int32Array([10, 20, 30, 40]);
out.map_add   = Array.from(mapI32(k, a, b, "add"));
out.map_mul   = Array.from(mapI32(k, a, b, "mul"));
out.scalar_add= Array.from(mapScalarI32(k, a, 7, "add"));
out.f64_add   = Array.from(mapF64(k, new Float64Array([1.5, 2.5]),
                                   new Float64Array([0.25, 0.25]), "add"));
// ten repeated calls into the same module: each returns its own result
out.repeated  = Array.from({length: 10}, (_, i) =>
    mapScalarI32(k, new Int32Array([i]), 100, "add")[0]);
out.gather    = Array.from(adjacencyGather(k,
    new Uint32Array([10, 11, 12, 13]),
    new Uint32Array([0, 2]), new Uint32Array([2, 4]), 4));
out.calls = out.repeated.length;
console.log(JSON.stringify(out));
'''

r = node_json(RUNS)
print("mapI32(a, b, 'add')      :", r["map_add"])
print("mapI32(a, b, 'mul')      :", r["map_mul"])
print("mapScalarI32(a, 7, 'add'):", r["scalar_add"])
print("mapF64(x, y, 'add')      :", r["f64_add"])
print("adjacencyGather(...)     :", r["gather"])

mapI32(a, b, 'add')      : [11, 22, 33, 44]
mapI32(a, b, 'mul')      : [10, 40, 90, 160]
mapScalarI32(a, 7, 'add'): [8, 9, 10, 11]
mapF64(x, y, 'add')      : [1.75, 2.75]
adjacencyGather(...)     : [10, 11, 12, 13]


In [7]:
print("ten repeated calls into ONE loaded module, each distinct:")
print("  ", r["repeated"])

# every result above is checkable by hand -- these are not timings, they are values
check = r["map_add"] == [11, 22, 33, 44] and r["map_mul"] == [10, 40, 90, 160]
check = check and r["scalar_add"] == [8, 9, 10, 11]
check = check and r["gather"] == [10, 11, 12, 13]
check = check and r["repeated"] == list(range(100, 110))
print("all hand-checkable:", check)

ten repeated calls into ONE loaded module, each distinct:
   [100, 101, 102, 103, 104, 105, 106, 107, 108, 109]
all hand-checkable: True


Integer lanes are **bit-exact** — `map_i32` add/sub/mul wrap mod 2³², and the
ABI table says so per symbol. `div`/`pow` round-trip through f64 and are not
bit-exact; that is recorded in `ABI['nf_map_i32'].note']` rather than left to be
discovered.

In [8]:
ABI_SRC = r'''
import { ABI } from "./dist/abi.js";
console.log(JSON.stringify({
  map_i32: ABI["nf_map_i32"], map_scalar_i32: ABI["nf_map_scalar_i32"],
  sssp_csr: ABI["nf_sssp_csr"], rc: Object.keys(ABI).length,
}));
'''
abi = node_json(ABI_SRC)
print("ABI table entries:", abi["rc"], "(one per wrapped kernel)")
for key in ("map_i32", "map_scalar_i32", "sssp_csr"):
    entry = abi[key]
    print()
    print(entry["symbol"])
    print("   wasm params :", entry["wasm"])
    print("   returns     :", entry["returns"])
    print("   codes       :", entry["codes"])

ABI table entries: 17 (one per wrapped kernel)

nf_map_i32
   wasm params : ['a', 'b', 'n', 'op', 'out']
   returns     : i32
   codes       : {'-1': 'null pointer', '-2': 'op code outside 0..=6 for this symbol'}

nf_map_scalar_i32
   wasm params : ['a', 'n', 's', 'op', 'out']
   returns     : i32
   codes       : {'-1': 'null pointer', '-2': 'op code outside 0..=6 for this symbol'}

nf_sssp_csr
   wasm params : ['indptr', 'np', 'indices', 'weights', 'e', 'source', 'dist']
   returns     : i32
   codes       : {'-1': 'null pointer', '-2': 'source out of range, or indptr not monotonic'}


**17 of 86 are wrapped.** The other **69** are reachable — named at the call
site — through `callRaw`, and are unwrapped *on purpose*: the pointer-vs-length
order of each argument is recorded nowhere in the repository, so a wrapper
written from the signature alone would be a guess dressed as an API.

In [9]:
COVER = r'''
import { WRAPPED, TOTAL_EXPORTS } from "./dist/index.js";
console.log(JSON.stringify({wrapped: [...WRAPPED], total: TOTAL_EXPORTS}));
'''
cov = node_json(COVER)
print(f"wrapped {len(cov['wrapped'])} of {cov['total']}")
for s in cov["wrapped"]:
    print("  ", s)
print(f"unwrapped, reachable by name through callRaw: {cov['total'] - len(cov['wrapped'])}")

wrapped 17 of 86
   nf_sssp_csr
   nf_sssp_csr_pred
   nf_sssp_batch
   nf_cost_travel_batch
   nf_cost_intern
   nf_rowwise_kway_time_argmin_gather
   nf_adjacency_slice
   nf_adjacency_gather
   nf_map_i32
   nf_map_scalar_i32
   nf_map_fscalar_i32
   nf_map_f32
   nf_map_f32_divpow
   nf_map_scalar_f32
   nf_map_scalar_f32_divpow
   nf_map_f64
   nf_map_scalar_f64
unwrapped, reachable by name through callRaw: 69


`callRaw` deliberately does **no** length validation and no BigInt coercion —
only the shared return-code table and trap conversion. That is why the README
calls it an escape hatch, and why `test/traps.test.mjs` asserts that a trap
leaves the memory it wrote intact. A kernel with no documented argument order is
reachable, not safe.

In [10]:
RAW = r'''
import { loadKernels, callRaw, Q_INF } from "./dist/index.js";
const k = await loadKernels();
const rc = callRaw(k, "nf_map_i32", 4, k.base, 0);   // name it, no wrapper
console.log(JSON.stringify({ rc, Q_INF }));
'''
raw = node_json(RAW)
print("callRaw(k, 'nf_map_i32', ...) returned:", raw["rc"])
print("Q_INF sentinel (unreachable / no speed):", raw["Q_INF"])

callRaw(k, 'nf_map_i32', ...) returned: 0
Q_INF sentinel (unreachable / no speed): 4294967295


## 4. Parity coverage, stated as the package states it

**9 of the 17** have Python parity fixtures. The eight graph kernels are wrapped
but **have no Python parity fixture yet**, and `test/parity.test.mjs` prints
their names on every run so the gap cannot be mistaken for coverage. That is
the honest count, and it is why this notebook demonstrates elementwise kernels
rather than the graph ones.

In [11]:
par = json.loads((TS / "test" / "fixtures" / "parity.json").read_text(
    encoding="utf-8"))
print("fixture file top-level keys:", sorted(par)[:8] if isinstance(par, dict) else type(par).__name__)
if isinstance(par, dict):
    for k, v in par.items():
        if isinstance(v, (list, dict)):
            print(f"  {k}: {len(v)} entries")
        else:
            print(f"  {k}: {v}")

fixture file top-level keys: ['cases', 'dll', 'generator', 'input_lcg', 'inputs', 'literal_max_n', 'note', 'numpy']
  cases: 203 entries
  dll: src/numfast/_native/numfast_native.dll
  generator: numfast-native/ts/test/gen_fixtures.py
  input_lcg: 4 entries
  inputs: 6 entries
  literal_max_n: 64
  note: reference (NumPy + the documented int32 round-trip) and native (ctypes into the committed DLL) were compared byte-for-byte before recording; a disagreement aborts this generator
  numpy: 2.5.1
  python: 3.14.6
  seed: 42
  tolerance: 2 entries
  tolerance_source: specs-rebuilt/conformance-profile.toml [tolerance]


## 5. What this notebook does not claim

* **No executor.** The `.wasm` is a set of column-at-a-time kernels. There is no
  planner, no query language and no scheduler inside it. The executor in the
  Python package is a Python planner and a Python CPU driver — the Rust layer is
  kernels either way.
* **No browser support.** The `.wasm` itself has zero imports and *would*
  instantiate in a browser, but the published npm package's only `"."` export
  statically imports `node:module`, `node:fs` and `node:crypto`. That is a
  **statically resolvable edge**, so bundling fails at build time, not at run
  time. `docs/INSTALL.md` states this as *not supported* rather than leaving it
  to be discovered.
* **86, not 85.** Older comments in this tree still say 85; the shipped bytes
  export 86 and `artifact.mjs` records why the count was moved. See
  `showcase/README.md`.